# BARAM 2026 — 최종 파이프라인 (v5)

**이 노트북 하나로 최종 제출(`submission_v5.csv`)을 end-to-end 재현한다.**

파이프라인 (실험 이력·채택 근거는 `README.md` §8 참조):
1. **Feature 65개** — 원 NWP lean(죽은 변수 31개 제거) + 물리 파생 + 파워커브 + 격자 공간·감률 (`wind_lib`)
2. **모델** — GBM(LightGBM+HistGBM 평균) 0.3 ⊕ MLP(256×3, 그룹 concat 임베딩, 3그룹 pooled, **시드 5개 앙상블**) 0.7
3. **FICR 후처리** — 학습기간 OOF로 debias(리드×풍속분위)·nudge(공식 FICR 최대화) fit → 그룹별 최적 조합 선택
4. **최종** — 2022~2024 전체 재학습 → 2025 예측 → [0, 설비용량] 클리핑 → 제출

검증 규율: 후처리 선택은 2024 holdout(2022-23 학습), 모든 보정기는 학습구간에서만 fit(누설 없음).

> ⚠️ macOS: torch+LightGBM OpenMP 충돌(segfault) 방지 가드가 첫 셀에 있음 — 제거 금지. MLP는 MPS 사용.
> 실행: `OMP_NUM_THREADS=1 uv run --with nbconvert --with ipykernel --with torch jupyter nbconvert --to notebook --execute --inplace PIPELINE_FINAL.ipynb`

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")     # torch+lightgbm segfault 가드 (import 전)
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)                          # 가드 2
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores, metric

DEV = "mps" if torch.backends.mps.is_available() else "cpu"
SEEDS = [15, 0, 1, 2, 3]                          # MLP 시드 앙상블
BLEND = 0.7                                       # MLP 가중 (v4 튜닝 + FiLM 실험 w-스캔 결과)
MLP_CFG = dict(h=256, depth=3, drop=0.0, lr=0.0015868563457489381, wd=1e-4, bs=256, emb=4)  # v4 random search 채택
GROUPS = (1, 2, 3)
print(f"device={DEV}  seeds={SEEDS}  blend w={BLEND}")

# 데이터: 검증된 preprocessed + v2 공간·감률 feature
FR, TGT, FR_TE = {}, {}, {}
for g in GROUPS:
    df, tgt = W.load_train(g); TGT[g] = tgt
    FR[g] = W.add_spatial(W.build(df, g), "train")
    FR_TE[g] = W.add_spatial(W.build(W.load_test(g), g), "test")
BASE_ALL = [c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS] + ["pc_pred_cf"]
FEATS = W.lean_features(BASE_ALL) + W.SPATIAL_COLS
print("features:", len(FEATS), "| train rows:", {g: len(FR[g]) for g in GROUPS})

device=mps  seeds=[15, 0, 1, 2, 3]  blend w=0.7


features: 65 | train rows: {1: 26199, 2: 26200, 3: 17537}


## 1. 모델 정의 (GBM 앙상블 + 시드앙상블 MLP 블렌드)

In [2]:
def lgbm(): return lgb.LGBMRegressor(objective="mae", n_estimators=600, learning_rate=0.03,
    num_leaves=63, min_child_samples=60, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_lambda=1.0, random_state=42, n_jobs=1, verbose=-1)      # n_jobs=1: 가드 3
def histgbm(): return HGB(loss="absolute_error", max_iter=600, learning_rate=0.03,
    max_leaf_nodes=63, l2_regularization=1.0, random_state=42)

def gbm_ens(tr, va, cols, tgt):
    lg = lgbm().fit(tr[cols], tr[tgt])
    hg = histgbm().fit(tr[cols].to_numpy(), tr[tgt].to_numpy())
    return 0.5 * (lg.predict(va[cols]) + hg.predict(va[cols].to_numpy()))

class MLP(nn.Module):
    def __init__(s, nf, ng=3, h=256, depth=3, drop=0.0, emb=4):
        super().__init__(); s.emb = nn.Embedding(ng, emb)
        layers = [nn.Linear(nf + emb, h), nn.GELU(), nn.Dropout(drop)]
        for _ in range(depth - 1): layers += [nn.Linear(h, h), nn.GELU(), nn.Dropout(drop)]
        layers += [nn.Linear(h, 1)]; s.net = nn.Sequential(*layers)
    def forward(s, x, g): return s.net(torch.cat([x, s.emb(g)], -1)).squeeze(-1)

def train_one(pool_tr, feats, seed):
    """pooled(3그룹) MLP 1개 학습. 시간순 뒤 15%로 early stopping."""
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr = pool_tr.sort_values("kst_dtm")
    mu = pool_tr[feats].mean(); sd = pool_tr[feats].std() + 1e-8
    X = ((pool_tr[feats] - mu) / sd).to_numpy(np.float32)
    y = pool_tr["cf"].to_numpy(np.float32); gid = pool_tr["gid"].to_numpy(np.int64)
    n = len(X); cut = int(n * 0.85)
    Xt = torch.tensor(X, device=DEV); yt = torch.tensor(y, device=DEV); gt = torch.tensor(gid, device=DEV)
    net = MLP(len(feats), **{k: MLP_CFG[k] for k in ("h", "depth", "drop", "emb")}).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=MLP_CFG["lr"], weight_decay=MLP_CFG["wd"])
    best = 1e9; st = None; pat = 0
    for ep in range(120):
        net.train(); perm = np.random.permutation(np.arange(cut))
        for i in range(0, len(perm), MLP_CFG["bs"]):
            b = torch.tensor(perm[i:i + MLP_CFG["bs"]], device=DEV)
            opt.zero_grad(); ((net(Xt[b], gt[b]) - yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl = (net(Xt[cut:], gt[cut:]) - yt[cut:]).abs().mean().item()
        if vl < best - 1e-5: best = vl; st = {k: v.clone() for k, v in net.state_dict().items()}; pat = 0
        else: pat += 1
        if pat >= 10: break
    net.load_state_dict(st); return net, (mu, sd)

def predict_one(net, scaler, fr, feats, g, cap):
    mu, sd = scaler
    X = torch.tensor(((fr[feats] - mu) / sd).to_numpy(np.float32), device=DEV)
    gid = torch.full((len(fr),), g - 1, dtype=torch.long, device=DEV)
    net.eval()
    with torch.no_grad(): p = net(X, gid).cpu().numpy()
    return np.clip(p, 0, 1) * cap

def blend_predict(tr_frames):
    """tr_frames: {g: (train_frame, predict_frame)} → {g: 블렌드 예측}. 누설 없음(학습은 train_frame만)."""
    pool = []
    for g, (tr2, _) in tr_frames.items():
        p = tr2[FEATS + ["kst_dtm"]].copy(); p["cf"] = tr2[TGT[g]] / W.CAP[g]; p["gid"] = g - 1; pool.append(p)
    pool = pd.concat(pool, ignore_index=True)
    mlp_acc = {g: [] for g in tr_frames}
    for sd_ in SEEDS:
        net, scaler = train_one(pool, FEATS, sd_)
        for g, (_, va2) in tr_frames.items():
            mlp_acc[g].append(predict_one(net, scaler, va2, FEATS, g, W.CAP[g]))
    out = {}
    for g, (tr2, va2) in tr_frames.items():
        cap = W.CAP[g]
        gp = np.clip(gbm_ens(tr2, va2, FEATS, TGT[g]), 0, cap)
        mp = np.mean(mlp_acc[g], axis=0)
        out[g] = np.clip((1 - BLEND) * gp + BLEND * mp, 0, cap)
    return out

## 2. 2024 holdout — 예측·OOF 생성 → FICR 후처리 선택

후처리 보정기(debias·nudge)는 **학습기간 OOF 예측**으로만 fit (2022↔2023 swap, 그룹3은 시간순 70/30).

In [3]:
tr_frames = {}
for g in GROUPS:
    tgt = TGT[g]; cap = W.CAP[g]; fr = FR[g]
    m_tr = fr.kst_dtm < W.VALID_START; m_va = fr.kst_dtm >= W.VALID_START
    iso = W.fit_powercurve(fr[m_tr], tgt, cap)          # 파워커브도 학습부분만 fit
    tr_frames[g] = (W.with_pc(fr[m_tr], iso), W.with_pc(fr[m_va], iso))
val_pred = blend_predict(tr_frames)
print("holdout 예측 완료")

OOF = {}
for g in GROUPS:
    tgt = TGT[g]; cap = W.CAP[g]; tr2 = tr_frames[g][0]
    oof = np.full(len(tr2), np.nan); years = sorted(tr2.kst_dtm.dt.year.unique())
    if len(years) >= 2:
        for ty in years:
            m_in = tr2.kst_dtm.dt.year != ty; m_out = (tr2.kst_dtm.dt.year == ty).to_numpy()
            oof[m_out] = blend_predict({g: (tr2[m_in], tr2[tr2.kst_dtm.dt.year == ty])})[g]
    else:
        n = len(tr2); cut = int(n * 0.7)
        oof[cut:] = blend_predict({g: (tr2.iloc[:cut], tr2.iloc[cut:])})[g]
    OOF[g] = oof
    print(f"group{g} OOF: {int(np.isfinite(oof).sum())}/{len(tr2)}")

holdout 예측 완료


group1 OOF: 17421/17421


group2 OOF: 17422/17422


group3 OOF: 2628/8759


In [4]:
def debias_fit(tr, tgt, oof):
    d = tr.assign(oof=oof); d = d[np.isfinite(d["oof"])].copy(); d["resid"] = d[tgt] - d["oof"]
    d["lb"] = pd.cut(d["lead_h"], bins=[15, 21, 27, 33, 40], labels=False, include_lowest=True)
    d["wq"] = pd.qcut(d["gfs_wind_speed_100m_mean"], 5, labels=False, duplicates="drop")
    return d.groupby(["lb", "wq"])["resid"].mean(), d["resid"].mean()

def debias_apply(va, pred, tbl, glob):
    v = va.copy()
    v["lb"] = pd.cut(v["lead_h"], bins=[15, 21, 27, 33, 40], labels=False, include_lowest=True)
    v["wq"] = pd.qcut(v["gfs_wind_speed_100m_mean"], 5, labels=False, duplicates="drop")
    return pred + np.array([tbl.get(k, glob) for k in zip(v["lb"], v["wq"])])

def nudge_fit(tr, tgt, oof, cap):
    """공식 FICR(발전량 가중 2티어)을 최대화하는 (scale, shift) 그리드 탐색."""
    d = tr.assign(oof=oof); d = d[np.isfinite(d["oof"])]
    yt = d[tgt].to_numpy(); yp = d["oof"].to_numpy(); best = (1.0, 0.0); bf = -1
    for s in np.linspace(0.90, 1.15, 26):
        for sh in np.linspace(-0.06, 0.06, 25) * cap:
            f = group_scores(yt, np.clip(yp * s + sh, 0, cap), cap)[1]
            if f > bf: bf = f; best = (s, sh)
    return best

STORE, choice = {}, {}
for g in GROUPS:
    tgt = TGT[g]; cap = W.CAP[g]; tr2, va2 = tr_frames[g]; bp = val_pred[g]
    tbl, glob = debias_fit(tr2, tgt, OOF[g]); s, sh = nudge_fit(tr2, tgt, OOF[g], cap)
    p1 = np.clip(debias_apply(va2, bp, tbl, glob), 0, cap)
    cand = {"P0_none": bp, "P1_debias": p1,
            "P3_nudge": np.clip(bp * s + sh, 0, cap),
            "P5_deb_nudge": np.clip(p1 * s + sh, 0, cap)}
    STORE[g] = dict(tbl=tbl, glob=glob, nudge=(s, sh))
    rows = []
    for name, p in cand.items():
        nm, fi = group_scores(va2[tgt].to_numpy(), p, cap)
        rows.append(dict(post=name, nmae=nm, ficr=fi, contrib=fi - nm))
    t = pd.DataFrame(rows).set_index("post"); choice[g] = t["contrib"].idxmax()
    print(f"group{g}: 선택 = {choice[g]}")
    print(t.round(4).to_string(), "\n")

def apply_post(g, frame, pred):
    cap = W.CAP[g]; st = STORE[g]; ch = choice[g]
    if ch == "P0_none": return pred
    if ch == "P1_debias": return np.clip(debias_apply(frame, pred, st["tbl"], st["glob"]), 0, cap)
    if ch == "P3_nudge": return np.clip(pred * st["nudge"][0] + st["nudge"][1], 0, cap)
    q = np.clip(debias_apply(frame, pred, st["tbl"], st["glob"]), 0, cap)
    return np.clip(q * st["nudge"][0] + st["nudge"][1], 0, cap)

ans = pd.DataFrame({f"kpx_group_{g}": tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
p_post = pd.DataFrame({f"kpx_group_{g}": apply_post(g, tr_frames[g][1], val_pred[g]) for g in GROUPS})
ts, omn, fi = metric(ans, p_post)
print(f"2024 holdout 공식 총점 = {ts:.4f}  (1-NMAE {omn:.4f}, FICR {fi:.4f})   [기대값 ≈ 0.646]")

group1: 선택 = P3_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1225  0.3336   0.2111
P1_debias     0.1231  0.3169   0.1938
P3_nudge      0.1146  0.4386   0.3240
P5_deb_nudge  0.1140  0.4244   0.3104 

group2: 선택 = P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1250  0.4197   0.2947
P1_debias     0.1246  0.4068   0.2822
P3_nudge      0.1238  0.4512   0.3274
P5_deb_nudge  0.1221  0.4510   0.3289 



group3: 선택 = P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1375  0.2840   0.1465
P1_debias     0.1380  0.2822   0.1442
P3_nudge      0.1454  0.3671   0.2218
P5_deb_nudge  0.1471  0.3712   0.2241 

2024 holdout 공식 총점 = 0.6461  (1-NMAE 0.8720, FICR 0.4203)   [기대값 ≈ 0.646]


## 3. 최종 학습(2022-2024 전체) → 2025 예측 → 제출 생성

In [5]:
full_frames = {}
for g in GROUPS:
    tgt = TGT[g]; cap = W.CAP[g]
    iso = W.fit_powercurve(FR[g], tgt, cap)             # 전체 학습기간 파워커브
    full_frames[g] = (W.with_pc(FR[g], iso), W.with_pc(FR_TE[g], iso))
test_pred = blend_predict(full_frames)

out = W.load_test(1)[["forecast_id", "kst_dtm"]].rename(columns={"kst_dtm": "forecast_kst_dtm"})
for g in GROUPS:
    out[f"kpx_group_{g}"] = apply_post(g, full_frames[g][1], test_pred[g])

# 제출 무결성 검사
assert out.shape[0] == 8760
for g in GROUPS:
    c = out[f"kpx_group_{g}"]
    assert (c >= 0).all() and (c <= W.CAP[g]).all() and c.notna().all()
out.to_csv("submission_v5.csv", index=False)
print("saved submission_v5.csv", out.shape)
print("예측 평균 이용률(%):", {g: round(100 * out[f"kpx_group_{g}"].mean() / W.CAP[g], 1) for g in GROUPS})

saved submission_v5.csv (8760, 5)
예측 평균 이용률(%): {1: np.float64(39.9), 2: np.float64(40.6), 3: np.float64(40.0)}


## 결과 메모

- 홀드아웃 총점 ≈ 0.646 (1-NMAE ≈ 0.872, FICR ≈ 0.42). 이는 2024 자가검증이며 2025 실전과 편차가 있을 수 있음(README §1 참조).
- 다음 개선 후보: 출력구간별 FICR nudge 세분화, group3 전용 보정.